# Predicción Inteligente de la Copa Mundial FIFA 2026

## Modelado y Simulación

En este notebook se desarrollará el modelo de Machine Learning encargado de predecir el resultado de un partido entre dos selecciones nacionales.

Posteriormente se utilizarán dichas predicciones para simular el resto del Mundial mediante técnicas de simulación Monte Carlo y estimar la probabilidad de que cada selección avance a las diferentes fases del torneo.

In [43]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(ROOT)
print("src" in [p.name for p in ROOT.iterdir()])

c:\Users\darie\Downloads\PROYECTO_FIFA2026
True


2026-07-08 00:41:39,076 - src.prediction - INFO - Modelo best_model.pkl cargado exitosamente.
2026-07-08 00:41:39,079 - src.prediction - INFO - Encoder label_encoder.pkl cargado exitosamente.
2026-07-08 00:41:39,265 - src.prediction - INFO - Columnas: ['grupo', 'matches', 'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'goal_difference', 'puntos', 'estado / fase actual', 'notas clave']
2026-07-08 00:41:39,266 - src.prediction - INFO - Primeros índices: ['mexico', 'sudafrica', 'corea del sur', 'republica checa', 'suiza', 'canada', 'bosnia y herzegovina', 'qatar', 'brasil', 'marruecos']
2026-07-08 00:41:39,266 - src.prediction - INFO - Dataset Dataset_Mundial_2026_Actualizado.xlsx cargado exitosamente.


{'home_team': 'Argentina',
 'away_team': 'Suiza',
 'winner': 'Argentina',
 'probabilities': {'Empate': 0.2572862325360791,
  'Local': 0.4838147477899013,
  'Visitante': 0.25889901967401996},
 'confidence': 48.38}

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

import joblib

In [2]:
#configuracion

ROOT_DIR = Path.cwd().parent

DATA_PROCESSED = ROOT_DIR / "data" / "processed"

MODELS = ROOT_DIR / "models"

In [3]:
#cargar dataset

model_df = pd.read_csv(
    DATA_PROCESSED / "dataset_modelo.csv"
)

model_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,Derrotas_Away,Goles_Favor_Away,Goles_Contra_Away,Diferencia_Goles_Away,Win_Rate_Away,Promedio_GF_Away,Promedio_GC_Away,Puntos_Historicos_Away,Es_Mundial_2026_Away,Resultado
0,2000-01-04,Egipto,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,2000,...,192,492.0,620.0,-128.0,0.315436,1.100671,1.387025,537,False,Local
1,2000-01-07,Túnez,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,2000,...,192,492.0,620.0,-128.0,0.315436,1.100671,1.387025,537,False,Local
2,2000-01-08,Trinidad and Tobago,Canadá,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,...,184,617.0,618.0,-1.0,0.381857,1.301688,1.303797,652,True,Empate
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False,2000,...,150,477.0,484.0,-7.0,0.348148,1.177778,1.195062,537,False,Empate
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True,2000,...,150,274.0,499.0,-225.0,0.238971,1.007353,1.834559,252,False,Empate


In [4]:
features = [

    "Partidos",

    "Victorias",

    "Empates",

    "Derrotas",

    "Goles_Favor",

    "Goles_Contra",

    "Win_Rate",

    "Promedio_GF",

    "Promedio_GC",

    "Partidos_Away",

    "Victorias_Away",

    "Empates_Away",

    "Derrotas_Away",

    "Goles_Favor_Away",

    "Goles_Contra_Away",

    "Win_Rate_Away",

    "Promedio_GF_Away",

    "Promedio_GC_Away"

]

In [5]:
X = model_df[features]

y = model_df["Resultado"]

In [6]:
#Codificar la variable objetivo
encoder = LabelEncoder()

y = encoder.fit_transform(y)


In [7]:
#Dividir Train/Test

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

In [9]:
#Predicciones

predictions = model.predict(X_test)

In [10]:
#Accuracy
accuracy = accuracy_score(

    y_test,

    predictions

)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5383


In [11]:
#Reporte

print(

    classification_report(

        y_test,

        predictions,

        target_names=encoder.classes_

    )

)

              precision    recall  f1-score   support

      Empate       0.29      0.23      0.26      1185
       Local       0.64      0.71      0.67      2449
   Visitante       0.52      0.50      0.51      1454

    accuracy                           0.54      5088
   macro avg       0.48      0.48      0.48      5088
weighted avg       0.52      0.54      0.53      5088



In [12]:
joblib.dump(
    model,
    MODELS / "random_forest_model.pkl"
)

['c:\\Users\\darie\\Downloads\\PROYECTO_FIFA2026\\models\\random_forest_model.pkl']

In [13]:
import sys
import sklearn
import xgboost

print("Python:", sys.version)
print("Scikit-Learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Scikit-Learn: 1.6.0
XGBoost: 2.1.0


In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [15]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent

DATA_PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"

# Modelado Predictivo

En este notebook se entrenarán y evaluarán distintos modelos de Machine Learning utilizando el dataset construido durante el proceso de Feature Engineering.

El objetivo es desarrollar un modelo capaz de predecir el resultado de un partido entre dos selecciones nacionales utilizando únicamente información disponible antes del encuentro.

Se iniciará con Random Forest como modelo base y posteriormente se comparará con otros algoritmos como XGBoost, LightGBM y CatBoost para seleccionar el modelo con mejor desempeño.

## Carga del dataset

Se carga el conjunto de datos generado en el Notebook anterior.

Este dataset contiene todas las variables históricas necesarias para iniciar el entrenamiento del modelo.

In [16]:
model_df = pd.read_csv(
    DATA_PROCESSED / "dataset_modelo_v2.csv"
)

print(model_df.shape)

model_df.head()

(25440, 51)


,date,home_team,away_team,tournament,neutral,target,home_matches,home_wins,home_draws,home_losses,...,win_rate_difference,goals_for_difference,goals_against_difference,is_world_cup,is_qualification,is_friendly,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches
0,2000-01-04,Egipto,Togo,Friendly,0,Local,0,0,0,0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
1,2000-01-07,Túnez,Togo,Friendly,0,Local,0,0,0,0,...,0.0,-1.0,-2.0,0,0,1,0,0,0,0
2,2000-01-08,Trinidad and Tobago,Canadá,Friendly,0,Empate,0,0,0,0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
3,2000-01-09,Burkina Faso,Gabon,Friendly,0,Empate,0,0,0,0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
4,2000-01-09,Guatemala,Armenia,Friendly,1,Empate,0,0,0,0,...,0.0,0.0,0.0,0,0,1,0,0,0,0


In [17]:
columns_to_drop = [

    "target",

    "date",

    "home_team",

    "away_team",

    "tournament"

]

X = model_df.drop(
    columns=columns_to_drop,
    errors="ignore"
)

y = model_df["target"]

print("Variables:", X.shape[1])
print("Registros:", X.shape[0])

X.head()

Variables: 46
Registros: 25440


,neutral,home_matches,home_wins,home_draws,home_losses,home_goals_for,home_goals_against,home_goal_difference,home_win_rate,home_avg_goals_for,...,win_rate_difference,goals_for_difference,goals_against_difference,is_world_cup,is_qualification,is_friendly,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches
0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
1,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,-1.0,-2.0,0,0,1,0,0,0,0
2,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
3,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
4,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,1,0,0,0,0


#Codificación de la variable objetivo
## Codificación de la variable objetivo

Los algoritmos de clasificación trabajan con valores numéricos.

Por ello la variable objetivo será convertida a una representación numérica utilizando LabelEncoder.


In [18]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

mapping = dict(
    zip(
        encoder.classes_,
        encoder.transform(encoder.classes_)
    )
)

print(mapping)

{'Empate': np.int64(0), 'Local': np.int64(1), 'Visitante': np.int64(2)}


## División de entrenamiento y prueba

El conjunto de datos será dividido en dos subconjuntos.

El 80 % será utilizado para entrenar el modelo mientras que el 20 % restante permitirá evaluar su capacidad para generalizar sobre partidos no vistos durante el entrenamiento.

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Entrenamiento:", X_train.shape)

print("Prueba:", X_test.shape)

Entrenamiento: (20352, 46)
Prueba: (5088, 46)


# Modelo Base - Random Forest

Random Forest será utilizado como primer modelo del proyecto.

Este algoritmo servirá para validar el pipeline de entrenamiento y establecer una línea base antes de evaluar modelos más avanzados.

In [20]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(

    n_estimators=500,

    max_depth=12,

    min_samples_split=5,

    min_samples_leaf=2,

    random_state=42,

    n_jobs=-1

)

rf_model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=12, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=500, n_jobs=-1, random_state=42)

## Predicciones

Una vez entrenado el modelo se generan predicciones sobre el conjunto de prueba para medir su rendimiento.

In [21]:
predictions = rf_model.predict(X_test)

## Exactitud del modelo

La exactitud representa el porcentaje de predicciones correctas realizadas sobre datos que el modelo nunca observó durante el entrenamiento.

In [22]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5825


In [23]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5825


## Reporte de clasificación

Se analizan las métricas de precisión, recall y F1-Score para cada una de las clases del problema con el objetivo de identificar fortalezas y oportunidades de mejora del modelo.

In [24]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predictions,
        target_names=encoder.classes_
    )
)

              precision    recall  f1-score   support

      Empate       0.34      0.02      0.04      1185
       Local       0.60      0.86      0.71      2449
   Visitante       0.56      0.56      0.56      1454

    accuracy                           0.58      5088
   macro avg       0.50      0.48      0.44      5088
weighted avg       0.53      0.58      0.51      5088



## Matriz de confusión

La matriz de confusión permite visualizar cómo se distribuyen las predicciones del modelo y cuáles son los errores más frecuentes entre las diferentes clases.

In [25]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    predictions
)

cm

array([[  26,  825,  334],
       [  27, 2117,  305],
       [  24,  609,  821]])

## Importancia de las variables

Random Forest permite estimar qué variables aportan mayor información durante el proceso de entrenamiento.

Este análisis facilita comprender qué factores tienen un mayor impacto en las predicciones realizadas por el modelo.

In [26]:
importance = (
    pd.DataFrame({
        "Variable": X.columns,
        "Importancia": rf_model.feature_importances_
    })
    .sort_values(
        by="Importancia",
        ascending=False
    )
)

importance.head(20)

,Variable,Importancia
38,goals_against_difference,0.096045
35,goal_difference_strength,0.069868
36,win_rate_difference,0.067219
37,goals_for_difference,0.047915
27,away_avg_goals_against,0.040128
10,home_avg_goals_against,0.039844
24,away_goal_difference,0.033246
25,away_win_rate,0.032386
7,home_goal_difference,0.030711
8,home_win_rate,0.028877


## Almacenamiento del modelo

Una vez validado el entrenamiento, el modelo será almacenado para reutilizarlo posteriormente durante la simulación del torneo y la aplicación desarrollada en Streamlit.

In [27]:
import joblib

joblib.dump(
    rf_model,
    MODELS / "random_forest_model.pkl"
)

joblib.dump(
    encoder,
    MODELS / "label_encoder.pkl"
)

print("Modelo almacenado correctamente.")

Modelo almacenado correctamente.


# Comparación de modelos

Una vez validado el funcionamiento del pipeline mediante Random Forest, se entrenarán distintos algoritmos de Machine Learning utilizando exactamente el mismo conjunto de datos.

El objetivo es comparar su desempeño bajo las mismas condiciones y seleccionar el modelo con mejor capacidad predictiva para la simulación del Mundial 2026.

## Modelo 1 - Logistic Regression

La regresión logística servirá como línea base para comparar el rendimiento de los demás modelos del proyecto.

In [28]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(

    max_iter=1000,

    random_state=42

)

lr_model.fit(
    X_train,
    y_train
)

lr_predictions = lr_model.predict(X_test)

lr_accuracy = accuracy_score(
    y_test,
    lr_predictions
)

print(f"Accuracy Logistic Regression: {lr_accuracy:.4f}")

Accuracy Logistic Regression: 0.5757


c:\Users\darie\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [29]:
results = pd.DataFrame({

    "Modelo":[
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy":[
        lr_accuracy,
        accuracy
    ]

})

results.sort_values(
    by="Accuracy",
    ascending=False
)

,Modelo,Accuracy
1,Random Forest,0.582547
0,Logistic Regression,0.575668


## Modelo 3 - XGBoost

XGBoost es uno de los algoritmos de Machine Learning más utilizados en problemas de clasificación debido a su capacidad para capturar relaciones complejas entre las variables y ofrecer un alto rendimiento predictivo.

In [30]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(

    n_estimators=500,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="mlogloss"

)

xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_predictions)

print(f"Accuracy XGBoost: {xgb_accuracy:.4f}")

Accuracy XGBoost: 0.5674


## Modelo 4 - LightGBM

LightGBM utiliza árboles de decisión optimizados para ofrecer tiempos de entrenamiento muy bajos sin sacrificar precisión.

In [31]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(

    n_estimators=500,

    learning_rate=0.05,

    random_state=42

)

lgbm_model.fit(X_train, y_train)

lgbm_predictions = lgbm_model.predict(X_test)

lgbm_accuracy = accuracy_score(y_test, lgbm_predictions)

print(f"Accuracy LightGBM: {lgbm_accuracy:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001562 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6306
[LightGBM] [Info] Number of data points in the train set: 20352, number of used features: 46
[LightGBM] [Info] Start training from score -1.456931
[LightGBM] [Info] Start training from score -0.731103
[LightGBM] [Info] Start training from score -1.252910
Accuracy LightGBM: 0.5684


## Modelo 5 - CatBoost

CatBoost es un algoritmo basado en Gradient Boosting que suele obtener excelentes resultados en conjuntos de datos tabulares y requiere muy poco preprocesamiento.

In [32]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(

    iterations=500,

    learning_rate=0.05,

    depth=6,

    random_seed=42,

    verbose=False

)

cat_model.fit(X_train, y_train)

cat_predictions = cat_model.predict(X_test)

cat_accuracy = accuracy_score(y_test, cat_predictions)

print(f"Accuracy CatBoost: {cat_accuracy:.4f}")

Accuracy CatBoost: 0.5818


## Comparación de modelos

Finalmente se comparan los algoritmos entrenados utilizando la misma división de entrenamiento y prueba.

El modelo con mayor desempeño será seleccionado para realizar la simulación de la Copa Mundial FIFA 2026.

In [33]:
results = pd.DataFrame({

    "Modelo":[
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],

    "Accuracy":[
        lr_accuracy,
        accuracy,
        xgb_accuracy,
        lgbm_accuracy,
        cat_accuracy
    ]

})

results = results.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

results

,Modelo,Accuracy
0,Random Forest,0.582547
1,CatBoost,0.581761
2,Logistic Regression,0.575668
3,LightGBM,0.568396
4,XGBoost,0.567414


In [34]:
results = pd.DataFrame({
    "Modelo": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "Accuracy": [
        lr_accuracy,
        accuracy,
        xgb_accuracy,
        lgbm_accuracy,
        cat_accuracy
    ]
})

# Ordenar de mayor a menor
results = results.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

# Transformar el 0.5851 a "58.51%"
results["Accuracy"] = results["Accuracy"].map("{:.2%}".format)

# Mostrar resultados
results

,Modelo,Accuracy
0,Random Forest,58.25%
1,CatBoost,58.18%
2,Logistic Regression,57.57%
3,LightGBM,56.84%
4,XGBoost,56.74%


In [35]:
best_model_name = results.iloc[0]["Modelo"]

print(f"Mejor modelo: {best_model_name}")

Mejor modelo: Random Forest


# Selección del modelo final

Después de entrenar y evaluar distintos algoritmos de Machine Learning, se seleccionó el modelo con mejor desempeño utilizando el mismo conjunto de entrenamiento y prueba.

El modelo seleccionado será utilizado durante la simulación de la Copa Mundial FIFA 2026 y posteriormente integrado en la aplicación desarrollada con Streamlit.

In [36]:
best_model = rf_model

best_model_name = "Random Forest"

print(f"Modelo seleccionado: {best_model_name}")

Modelo seleccionado: Random Forest


In [37]:
import joblib

joblib.dump(
    best_model,
    MODELS / "best_model.pkl"
)

joblib.dump(
    encoder,
    MODELS / "label_encoder.pkl"
)

print("Modelo guardado correctamente.")

Modelo guardado correctamente.


In [46]:
#PRUEBAA LUEGO DE HACER EL PREDICTION PY Y EL SIMULATION PY

from src.prediction import load_world_cup_data

world = load_world_cup_data()

type(world)


2026-07-08 00:41:56,104 - src.prediction - INFO - Columnas: ['grupo', 'matches', 'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'goal_difference', 'puntos', 'estado / fase actual', 'notas clave']
2026-07-08 00:41:56,105 - src.prediction - INFO - Primeros índices: ['mexico', 'sudafrica', 'corea del sur', 'republica checa', 'suiza', 'canada', 'bosnia y herzegovina', 'qatar', 'brasil', 'marruecos']
2026-07-08 00:41:56,105 - src.prediction - INFO - Dataset Dataset_Mundial_2026_Actualizado.xlsx cargado exitosamente.


pandas.core.frame.DataFrame

In [47]:
from src.prediction import predict_match

In [48]:
from src.prediction import predict_match

predict_match("Francia", "Marruecos")

{'home_team': 'Francia',
 'away_team': 'Marruecos',
 'winner': 'Francia',
 'probabilities': {'Empate': 0.271105593697392,
  'Local': 0.49384317048558024,
  'Visitante': 0.2350512358170277},
 'confidence': 49.38}

In [49]:
predict_match("España", "Bélgica")

{'home_team': 'España',
 'away_team': 'Bélgica',
 'winner': 'España',
 'probabilities': {'Empate': 0.24439968541550894,
  'Local': 0.49829103168836675,
  'Visitante': 0.2573092828961246},
 'confidence': 49.83}

In [50]:
predict_match("Inglaterra", "Noruega")

{'home_team': 'Inglaterra',
 'away_team': 'Noruega',
 'winner': 'Inglaterra',
 'probabilities': {'Empate': 0.28113270763055576,
  'Local': 0.4955721811486785,
  'Visitante': 0.22329511122076592},
 'confidence': 49.56}

In [51]:
predict_match("Argentina", "Suiza")

{'home_team': 'Argentina',
 'away_team': 'Suiza',
 'winner': 'Argentina',
 'probabilities': {'Empate': 0.257286232536079,
  'Local': 0.4838147477899014,
  'Visitante': 0.25889901967401985},
 'confidence': 48.38}

In [52]:
from src.simulation import simulate_world_cup

results = simulate_world_cup(simulations=10000)

2026-07-08 00:42:34,707 - src.simulation - INFO - Iniciando simulación Monte Carlo (10000 iteraciones)...
Simulando Mundiales: 100%|██████████| 10000/10000 [00:02<00:00, 4718.73it/s]
2026-07-08 00:42:36,854 - src.simulation - INFO - Reporte automatizado MD guardado en C:\Users\darie\Downloads\PROYECTO_FIFA2026\outputs\reports\tournament_report.md
2026-07-08 00:42:36,855 - src.simulation - INFO - Simulación finalizada exitosamente.


In [56]:
from src.prediction import get_team_features

print(get_team_features("Argentina"))

print()

print(get_team_features("Francia"))

grupo                                                   J
matches                                                 5
wins                                                    5
draws                                                   0
losses                                                  0
goals_for                                              13
goals_against                                           2
goal_difference                                        11
puntos                                                 15
estado / fase actual       Clasificado a Cuartos de Final
notas clave             Venció a Egipto hoy 7/7. Invicto.
Name: argentina, dtype: object

grupo                                                I
matches                                              5
wins                                                 4
draws                                                1
losses                                               0
goals_for                                           11


In [57]:
from src.simulation import simulate_world_cup

results = simulate_world_cup(
    simulations=10000
)

2026-07-08 00:45:12,757 - src.simulation - INFO - Iniciando simulación Monte Carlo (10000 iteraciones)...
Simulando Mundiales: 100%|██████████| 10000/10000 [00:02<00:00, 4661.43it/s]
2026-07-08 00:45:14,911 - src.simulation - INFO - Reporte automatizado MD guardado en C:\Users\darie\Downloads\PROYECTO_FIFA2026\outputs\reports\tournament_report.md
2026-07-08 00:45:14,912 - src.simulation - INFO - Simulación finalizada exitosamente.
